# Indexing And Retrieval

This notebook focuses on how indexed payloads, chunking, filtering, and paper-scoped retrieval work together.


## Setup And Demo Papers

The example uses the same two small papers as notebook 01, now taken from the
shared `episcope_nb` helper, so every retrieved chunk can be inspected directly.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Locate `notebooks/episcope_nb.py`, the shared helper module. This works whether
# the kernel starts in `notebooks/` or at the repository root.
_cwd = Path.cwd()
_nb_dir = next(
    (
        directory
        for candidate in [_cwd, *_cwd.parents]
        for directory in (candidate, candidate / "notebooks")
        if (directory / "episcope_nb.py").is_file()
    ),
    None,
)
if _nb_dir is None:
    raise FileNotFoundError("Could not find notebooks/episcope_nb.py")
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

# Importing the helper also puts `src/` on sys.path when this is a checkout.
import episcope_nb as nb

WORK_DIR = nb.bootstrap()
WORK_DIR

In [ ]:
sample_papers = nb.sample_papers()
sample_papers["paper_open_data"]["metadata"].title, list(sample_papers)

In [ ]:
embedder = nb.TinyKeywordEmbedder()
embedder.model_name, embedder.dim

## Build A Local Index

Chunking controls what the retriever can return. Shorter chunks are easier to inspect; larger chunks preserve more surrounding context.


In [ ]:
from episcope.rag.indexing.chunking import FixedSizeChunker, ParagraphChunker

sample_text = sample_papers["paper_open_data"]["sections"][1].content
chunkers = {
    "fixed_size": FixedSizeChunker(chunk_size=90, chunk_overlap=20),
    "paragraph": ParagraphChunker(min_chunk_size=40),
}

for name, chunker in chunkers.items():
    chunks = chunker.chunk(sample_text)
    print(f"{name}: {len(chunks)} chunk(s)")
    for chunk in chunks:
        if len(chunk.split()) < 4:
            continue
        preview = chunk[:85].rsplit(" ", 1)[0]
        if preview != chunk:
            preview = preview + "..."
        print("  -", preview)


In [ ]:
from episcope.rag.indexing.chunking import FixedSizeChunker
from episcope.rag.indexing.indexer import Indexer
from episcope.rag.retrieval.candidates import SemanticCandidateRetriever
from episcope.rag.retrieval.retriever import Retriever
from episcope.vectordb.file import FileDB

vdb = FileDB(str(WORK_DIR / "index"))
indexer = Indexer(
    vdb,
    embedder=embedder,
    chunker=FixedSizeChunker(chunk_size=500, chunk_overlap=50),
)

for paper_id, paper in sample_papers.items():
    indexer.index_paper(paper["sections"], paper["metadata"], paper_id=paper_id)

vdb.save()
semantic_candidates = SemanticCandidateRetriever(vdb, dense_embedder=embedder)
retriever = Retriever(vdb, candidate_retrievers=[semantic_candidates], use_rerank=False)

len(vdb.get_points()), vdb.get_embedding_model()


## Inspect Indexed Payloads

Each vector point keeps the text plus useful metadata such as `paper_id`, `section_title`, and `section_type`. These fields support filtering and provenance.


In [ ]:
for point in vdb.get_points()[:4]:
    print({key: point.get(key) for key in ["paper_id", "section_title", "section_type", "is_metadata"]})
    print(point["text"][:160], "\n")


## Compare Corpus And Paper-Scoped Retrieval

Use `retrieve` to search the whole index and `retrieve_by_paper` when a workflow should only use evidence from one paper.


In [ ]:
query = "patient-level registry data source"
corpus_results = retriever.retrieve(query, top_k=4)
paper_results = retriever.retrieve_by_paper(query, "paper_closed_data", top_k=3)

print("Corpus search")
for result in corpus_results:
    print(f"- {result.paper_id}: {result.section_title} ({result.similarity_score:.3f})")

print("\nWithin paper_closed_data")
for result in paper_results:
    print(f"- {result.paper_id}: {result.section_title} ({result.similarity_score:.3f})")


## Filter By Payload Fields

Filters are useful when a task should search only methods, data availability statements, abstracts, or another payload-defined slice.


In [ ]:
filtered = retriever.retrieve(
    "can the dataset be shared publicly?",
    top_k=5,
    filter={"section_type": "Data availability"},
)

for result in filtered:
    print(f"- {result.paper_id}: {result.section_title}")
    print(result.text[:220], "\n")
